## Dataset: Robos de Automotores, Armas de Fuego y Denuncias de Robo (2018-2025)

**Fuente:** Ministerio de Interior y Policía (MIP), vía datos.gob.do
https://datos.gob.do/dataset/estadisticas-de-robos-de-automotores-armas-de-fuego-y-denuncias-de-robos

**Licencia:** Open Data Commons Open Database License (ODbL)

**Nota sobre la descarga:** el archivo original viene en formato Excel con 3 hojas
(ROBOS_AUTOMOTOR, ROBOS_ARMAS_FUEGO, ROBOS_GENERALES), cada una con un esquema de
columnas distinto. Se encapsuló la unificación en `build_dataset_from_excel()`
dentro de `src/inf8239_u01/data.py`, que combina las 3 hojas en un solo dataset
tabular y agrega la columna `categoria_robo` (Automotor / Armas de Fuego / General)
como variable de origen.

In [ ]:
import sys
from pathlib import Path

# Asegura que 'src' esté en sys.path sin importar el cwd del kernel
_root = Path.cwd()
while not (_root / "src").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root / "src"))

In [3]:
from inf8239_u01.data import build_dataset_from_excel
path = build_dataset_from_excel(
    "data/raw/robos-automotor-armas-de-fuego-y-denuncias-de-robo-2018-2025.xlsx"
)
print(path)

C:\INF8239_U01\data\raw\dataset.csv


## Paso 5: Cargar y reconocer el esquema


In [4]:
import pandas as pd

# Se usa _root (definido en la celda anterior) en vez de una ruta relativa
# suelta, porque el cwd del kernel de Jupyter es la carpeta "notebooks",
df = pd.read_csv(_root / "data/raw/dataset.csv")
print(df.shape)
print(df.dtypes)
print(df.head())
print(df.tail())
assert not df.empty


(6516, 6)
anio                  int64
mes                     str
provincia               str
categoria_robo          str
tipo_detalle            str
cantidad_denuncias    int64
dtype: object
   anio  mes          provincia categoria_robo                   tipo_detalle  \
0  2018  abr            Baoruco      Automotor               Robo Motocicleta   
1  2018  abr           Barahona      Automotor               Robo Motocicleta   
2  2018  abr            Dajabón      Automotor               Robo Motocicleta   
3  2018  abr  Distrito Nacional      Automotor  Robo Vehículos 4 ruedas y más   
4  2018  abr  Distrito Nacional      Automotor               Robo Motocicleta   

   cantidad_denuncias  
0                   6  
1                   5  
2                   1  
3                  11  
4                  48  
      anio  mes      provincia categoria_robo         tipo_detalle  \
6511  2025  sep  Santo Domingo        General    Rotura A Vehículo   
6512  2025  sep       Valverde       

## Paso 6: Construir la auditoria


In [5]:
audit = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "ausentes": df.isna().sum(),
    "porcentaje_ausente": (df.isna().mean()*100).round(2),
    "unicos": df.nunique(dropna=False)
}).sort_values("porcentaje_ausente", ascending=False)
print("Duplicados:", df.duplicated().sum())
display(audit)


Duplicados: 0


,tipo,ausentes,porcentaje_ausente,unicos
tipo_detalle,str,1061,16.28,11
anio,int64,0,0.00,8
mes,str,0,0.00,12
provincia,str,0,0.00,35
categoria_robo,str,0,0.00,3
cantidad_denuncias,int64,0,0.00,314


## Paso 7: Definir target y retirar fugas

**Target:** `categoria_robo` (3 clases: Automotor, Armas de Fuego, General).

**Columna retirada:** `tipo_detalle`. Justificacion (no es por metrica, es por fuga de informacion):
los valores de `tipo_detalle` son subtipos que existen unicamente dentro de una sola categoria
(por ejemplo, "Robo Motocicleta" solo aparece en filas de Automotor; "Arrebato", "Asalto (Atraco)",
"Robo Simple" y "Rotura (Escalamiento)" solo aparecen en General), y para Armas de Fuego la columna
queda completamente vacia. Esto convierte a `tipo_detalle` en un proxy casi perfecto del target: un
modelo que la use no aprenderia un patron real, simplemente memorizaria a que categoria pertenece
cada subtipo. Por eso se excluye de X.


In [ ]:
TARGET = "categoria_robo"
DROP_COLUMNS = ["tipo_detalle"]  
assert TARGET in df.columns

X = df.drop(columns=[TARGET] + DROP_COLUMNS)
y = df[TARGET]

print(y.value_counts(dropna=False))
assert y.notna().all()
assert y.nunique() >= 2


categoria_robo
General           2964
Automotor         2491
Armas de Fuego    1061
Name: count, dtype: int64


## Paso 8: Separar tipos y crear preprocesamiento


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                     ("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocess = ColumnTransformer([("num", num_pipe, num_cols),
                                 ("cat", cat_pipe, cat_cols)])
print(num_cols, cat_cols)


['anio', 'cantidad_denuncias'] ['mes', 'provincia']


## Paso 9: Dividir, baseline y SVM

> **Nota:** este es el entrenamiento final del modelo.


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

dummy = Pipeline([("prep", preprocess), ("model", DummyClassifier(strategy="most_frequent"))])
svm = Pipeline([("prep", preprocess), ("model", SVC(C=1, gamma="scale", probability=True, random_state=42))])

for name, model in {"dummy": dummy, "svm": svm}.items():
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    print(name, f1_score(yte, pred, average="macro"))

print(classification_report(yte, svm.predict(Xte)))


dummy 0.20839922684941137


c:\INF8239_U01\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


svm 0.5515050166115025
                precision    recall  f1-score   support

Armas de Fuego       0.42      0.05      0.08       212
     Automotor       0.68      0.71      0.69       499
       General       0.78      1.00      0.88       593

      accuracy                           0.73      1304
     macro avg       0.63      0.58      0.55      1304
  weighted avg       0.68      0.73      0.68      1304

